In [ ]:
!pip install feedparser faiss-cpu symspellpy

In [ ]:
import feedparser
import urllib.request as libreq
import pandas as pd

Скачиваем данные о научных публикациях через API arxiv

In [ ]:
for maximum in range(1000, 2000):
  url = f"http://export.arxiv.org/api/query?search_query=all:machine+learning&start=0&max_results={maximum}"
  with libreq.urlopen(url) as url:
    r = url.read()
  docs = feedparser.parse(r)
  titles = [d["title"] for d in docs["entries"]]
  print(len(set(titles)))
  if len(set(titles)) >= 1000:
    break
docs["entries"] = docs["entries"][:1000]
print("Finally: " + str(len(set(titles))))
for title in titles:
  print(title)

1000
Finally: 1000
Changing Data Sources in the Age of Machine Learning for Official Statistics
DOME: Recommendations for supervised machine learning validation in biology
Learning Curves for Decision Making in Supervised Machine Learning: A Survey
Active learning for data streams: a survey
Physics-Inspired Interpretability Of Machine Learning Models
Privacy-preserving machine learning for healthcare: open challenges and future perspectives
A Benchmark Study of Machine Learning Models for Online Fake News Detection
Emotion in Reinforcement Learning Agents and Robots: A Survey
MEMe: An Accurate Maximum Entropy Method for Efficient Approximations in Large-Scale Machine Learning
Generalizing Machine Learning Evaluation through the Integration of Shannon Entropy and Rough Set Theory
ALERT-Transformer: Bridging Asynchronous and Synchronous Machine Learning for Real-Time Event-based Spatio-Temporal Data
Learning Representations from Dendrograms
Public Policymaking for International Agricultu

Выделяем из данных названия (title), авторов (author) и аннотации (abstract) публикаций

In [ ]:
pubs = []
for entry in docs["entries"]:
  data = {"title": entry['title'],
          "authors": [author['name'] for author in entry['authors']],
          "abstract": entry['summary']}
  pubs.append(data)

df = pd.DataFrame(pubs)
df

,title,authors,abstract
0,Changing Data Sources in the Age of Machine Le...,"[Cedric De Boom, Michael Reusens]",Data science has become increasingly essential...
1,DOME: Recommendations for supervised machine l...,"[Ian Walsh, Dmytro Fishman, Dario Garcia-Gasul...",Modern biology frequently relies on machine le...
2,Learning Curves for Decision Making in Supervi...,"[Felix Mohr, Jan N. van Rijn]",Learning curves are a concept from social scie...
3,Active learning for data streams: a survey,"[Davide Cacciarelli, Murat Kulahci]",Online active learning is a paradigm in machin...
4,Physics-Inspired Interpretability Of Machine L...,"[Maximilian P Niroomand, David J Wales]",The ability to explain decisions made by machi...
...,...,...,...
995,Deep Learning for Classical Japanese Literature,"[Tarin Clanuwat, Mikel Bober-Irizar, Asanobu K...",Much of machine learning research focuses on p...
996,One-Shot Federated Learning,"[Neel Guha, Ameet Talwalkar, Virginia Smith]","We present one-shot federated learning, where ..."
997,Improving Meta-Learning Generalization with Ac...,"[Simon Guiroy, Christopher Pal, Gonçalo Mordid...",Meta-Learning algorithms for few-shot learning...
998,LeanML: A Design Pattern To Slash Avoidable Wa...,[Yves-Laurent Kom Samo],We introduce the first application of the lean...


Разделяем тексты аннотаций на чанки по 200 слов с перекрытием в 30 слов, так как во всех данных аннотациях статей меньше 500 слов, значит и размеры чанков с перекрытиями надо уменьшить

In [ ]:
def split_chunks(text, chunk_size=200, overlap=30):
  chunks = []
  words = text.split()

  start = 0
  end = 0
  while start < len(words) and end < len(words):
    end = min(start + chunk_size, len(words))
    chunks.append(" ".join(words[start:end]))
    start = start + overlap

  return chunks

In [ ]:
total_chunks = []
publications = [pub["abstract"] for pub in pubs]

metadata = []
for publication in publications:
  pub_chunks = split_chunks(publication)
  total_chunks.extend(pub_chunks)

print(f"Общее количество чанков: {len(total_chunks)}")
print(total_chunks[346])

Общее количество чанков: 1349
In this paper, a Wide Learning architecture is proposed that attempts to automate the feature engineering portion of the machine learning (ML) pipeline. Feature engineering is widely considered as the most time consuming and expert knowledge demanding portion of any ML task. The proposed feature recommendation approach is tested on 3 healthcare datasets: a) PhysioNet Challenge 2016 dataset of phonocardiogram (PCG) signals, b) MIMIC II blood pressure classification dataset of photoplethysmogram (PPG) signals and c) an emotion classification dataset of PPG signals. While the proposed method beats the state of the art techniques for 2nd and 3rd dataset, it reaches 94.38% of the accuracy level of the winner of PhysioNet Challenge 2016. In all cases, the effort to reach a satisfactory performance was drastically less (a few days) than manual feature engineering.


Теперь преобразуем тексты в эмбеддинги и построим векторное хранилище

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
vectorize_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = vectorize_model.encode(total_chunks, show_progress_bar=True)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Настроим взаимодействие с LLM FLAN-T5 от Google

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
llm.to("cuda")

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
def generate_answer(prompt):
  inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)
  inputs.to("cuda")
  outputs = llm.generate(**inputs, max_new_tokens=300, top_p=0.9, temperature=0.3)
  return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def generate_answer_with_RAG(prompt, chunks):
  context = "\n\n".join(chunks)
  rag_prompt = f"Role: machine learning expert. Use next parts of context to answer the question. Context: {context}. Question: {prompt}"
  return generate_answer(rag_prompt)

Реализуем поиск по корпусу

In [ ]:
def corpus_search(query, chunks, top_k=5):
  results = []
  query_embedding = vectorize_model.encode([query])
  similarity_scores, indices = index.search(query_embedding, top_k)
  results = [chunks[i] for i in indices[0]]

  return results

Добавим обработку опечаток в запросах с испольованием SymSpell

In [ ]:
import pkg_resources
from symspellpy import SymSpell

/tmp/ipython-input-3427826714.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [ ]:
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = pkg_resources.resource_filename("symspellpy", "frequency_dictionary_en_82_765.txt")
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)

True

In [ ]:
def fix_query(query):
  corrections = sym_spell.lookup_compound(query, max_edit_distance=2)
  return corrections[0].term if corrections else query

Протестируем вышенаписанный код на 5 промптах

In [ ]:
def test_rag(query, chunks):
  context_chunks = corpus_search(query, chunks)
  return generate_answer_with_RAG(query, context_chunks)

In [ ]:
queries = [
    "Whot is parallelism of machine learning?",
    "Which benefit of using deeeep learning for weather?",
    "What is back propogation algoritm?",
    "How machine learnin can help to find cancer tumors?",
    "What is clasification model for ecology problems?",
]

for query in queries:
  fixed_query = fix_query(query)
  fixed_query = fixed_query + "?"
  print(f"Query: {fixed_query}\n")
  print("Without RAG:\n", generate_answer(fixed_query), "\n")
  print("With RAG:\n", test_rag(fixed_query, total_chunks), "\n")

Query: what is parallelism of machine learning?

Without RAG:
 parallel 

With RAG:
 Multivariate Learning. 

Query: which benefit of using deep learning for weather?

Without RAG:
 a better understanding of the weather 

With RAG:
 We propose a neural network architecture that can predict urban land surface processes using a combination of ML and deep learning. 

Query: what is back propagation algorithm?

Without RAG:
 back propagation 

With RAG:
 Learning Multi-Index Functions in Two-layer Neural Networks 

Query: how machine learning can help to find cancer tutors?

Without RAG:
 a machine learning algorithm 

With RAG:
 We compare the accuracy of the ML algorithms on the Wisconsin Diagnostic Breast Cancer dataset by comparing their classification test accuracy and their sensitivity and specificity values. 

Query: what is classification model for ecology problems?

Without RAG:
 a phylogenetic model 

With RAG:
 Bioclimatic models are a key tool for predicting the range of organi

Видим, что LLM с RAG дает более конкретные и полные ответы, чем без него. Значит, RAG действительно улучшает качество ответов LLM